# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SammyFarrelZebua/flyrank-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Lane: CTR / Engagement Opportunity Scoring

We build a clean feature vector for predicting expected CTR per content item per client. Key engineering decisions:

- **Filter**: Remove rows where `avg_position == 0` (no GSC rank data — 1,205 rows). These cannot be evaluated against any position-aware CTR benchmark.
- **Missing-value flags first**: For `word_count` (25.7% missing) and `search_volume` (8.2% missing), add binary `has_*` flags *before* imputing with 0. Without flags, `fillna(0)` silently encodes `content_type` into a numeric column — a systematic bias.
- **Log-transform impressions**: `impressions_90d` is right-skewed; `log1p` compresses the scale and prevents the model from being dominated by extreme values.
- **Categorical encoding**: `content_type` is one-hot encoded (drop-first to avoid multicollinearity).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Load data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# 2. Filter invalid rows (no position data)
df_clean = df[df['avg_position'] > 0].copy()

# 3. Handle Systematic Missingness via Flags BEFORE imputation
df_clean['has_word_count'] = (~df_clean['word_count'].isnull()).astype(int)
df_clean['has_search_volume'] = (~df_clean['search_volume'].isnull()).astype(int)

# Safe Zero Imputation for numeric columns now that flags exist
df_clean['word_count'] = df_clean['word_count'].fillna(0)
df_clean['search_volume'] = df_clean['search_volume'].fillna(0)

# 4. Feature Engineering
# Log-transform impressions to handle heavy right-skew
df_clean['log_impressions_90d'] = np.log1p(df_clean['impressions_90d'])

# 5. Categorical Encoding (content_type)
df_encoded = pd.get_dummies(df_clean, columns=['content_type'], drop_first=True, dtype=int)

# 6. Final Feature Selection
target_col = 'ctr'
context_cols = ['content_id', 'client_id']

# Capture all one-hot encoded columns automatically
type_cols = [c for c in df_encoded.columns if c.startswith('content_type_')]

feature_cols = [
    'avg_position', 
    'log_impressions_90d', 
    'word_count', 'has_word_count',
    'search_volume', 'has_search_volume',
    'content_age_days'
] + type_cols

# Compile final vector
feature_vector = df_encoded[context_cols + feature_cols + [target_col]].copy()

print(f"Original rows: {len(df):,} | Processed rows: {len(feature_vector):,}")
print(f"Features: {len(feature_cols)}")
print(f"Context: {len(context_cols)}")
print("Target: 1")
print("\nFeature Vector Head:")
display(feature_vector.head(3))

Original rows: 30,000 | Processed rows: 28,795
Features: 9
Context: 2
Target: 1

Feature Vector Head:


,content_id,client_id,avg_position,log_impressions_90d,word_count,has_word_count,search_volume,has_search_volume,content_age_days,content_type_feedly article,content_type_keyword article,ctr
0,content_304f48230142,client_f369cb89fc,10.6,8.243808,3221.0,1,10.0,1,187,0,1,0.76
1,content_a1fb4e703a9e,client_4e07408562,20.3,9.636980,2481.0,1,90.0,1,445,0,1,0.05
2,content_9aa793d4d895,client_7f2253d7e2,36.5,9.440023,3515.0,1,0.0,1,141,0,1,0.09


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Definition & Availability Matrix

| Feature | Type | Meaning | Missingness Treatment | Available-when? Justification |
|---|---|---|---|---|
| `avg_position` | Numeric | Average daily ranking position in SERP | None (filtered out 1,205 invalid 0 values) | **Before prediction**: GSC tracks position *prior* to evaluation window. |
| `log_impressions_90d` | Numeric | Log-transformed trailing 90-day search impressions | None (all rows have impressions > 0) | **Before prediction**: Search demand volume is logged historically before the evaluation date. |
| `word_count` | Numeric | Length of the published content in words | Zero-imputed (complemented by binary flag) | **Before prediction**: Page content length is static and established prior to search activity. |
| `has_word_count` | Binary | Flag indicating if page has text depth data | Categorical missingness captured as 0 | **Before prediction**: Knowable prior to evaluation based on page crawl history. |
| `search_volume` | Numeric | Target keyword monthly search volume | Zero-imputed (complemented by binary flag) | **Before prediction**: Search index keyword demand is derived from historical index data. |
| `has_search_volume` | Binary | Flag indicating if page has associated keyword metadata | Categorical missingness captured as 0 | **Before prediction**: Keyword index tracking is set up before evaluation. |
| `content_age_days` | Numeric | Days since initial publication | None (all rows have age >= 90 days) | **Before prediction**: Publication date is fixed historically. |
| `content_type_*` | Binary | One-hot encoded page categories (drop-first) | None (fully populated) | **Before prediction**: Page structural template is set prior to evaluation. |

---

### Verification of Feature Frame Missingness and Dtypes

We execute a quick diagnostic on our constructed feature vector to verify dtypes and lack of nulls.

In [2]:
# Verify that feature_vector has zero null values and proper numeric types
null_counts = feature_vector[feature_cols].isnull().sum()
print("Null count across all features:")
print(null_counts[null_counts > 0] if (null_counts > 0).any() else "✓ PASS: 0 null values across all features.")

print(f"\nTarget Variable ('ctr') stats:")
print(f"  Min:  {feature_vector['ctr'].min():.4f}%")
print(f"  Max:  {feature_vector['ctr'].max():.2f}%")
print(f"  Mean: {feature_vector['ctr'].mean():.4f}%")
print(f"  MAE of predicting overall mean (naive benchmark): {np.abs(feature_vector['ctr'] - feature_vector['ctr'].mean()).mean():.4f}%")

Null count across all features:
✓ PASS: 0 null values across all features.

Target Variable ('ctr') stats:
  Min:  0.0000%
  Max:  100.00%
  Mean: 0.5197%
  MAE of predicting overall mean (naive benchmark): 0.7095%


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Active Leakage Audit & Validation Design

To rigorously audit feature leakage, we design an experiment with **GroupKFold cross-validation on `client_id`**. Split by client ensures our model generalizes across different client sites rather than memorizing client-specific baseline CTRs.

We perform a 3-step test:
1. **Honest Model**: Evaluate `RandomForestRegressor` on clean features using 5-Fold Client Split. Record baseline Mean Absolute Error (MAE).
2. **Attack with Leaky Feature (`clicks_90d`)**: `ctr = (clicks_90d / impressions_90d) * 100`. Injecting `clicks_90d` introduces direct target numerator leakage.
3. **Purge & Confirm**: Remove `clicks_90d`, verify MAE returns to honest levels, and document the fake score jump.

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

# Set up 5-Fold Group Split on client_id
gkf = GroupKFold(n_splits=5)

X_honest = feature_vector[feature_cols]
y = feature_vector['ctr']
groups = feature_vector['client_id']

# ── 1. HONEST MODEL EVALUATION (GroupKFold across clients) ───────────────────
honest_maes = []
for train_idx, val_idx in gkf.split(X_honest, y, groups):
    X_tr, y_tr = X_honest.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X_honest.iloc[val_idx], y.iloc[val_idx]
    
    rf = RandomForestRegressor(n_estimators=30, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    preds = rf.predict(X_va)
    honest_maes.append(mean_absolute_error(y_va, preds))

mae_honest_cv = np.mean(honest_maes)
print(f"1. HONEST MODEL (GroupKFold MAE):  {mae_honest_cv:.4f}%")

# ── 2. ATTACK WITH LEAKY FEATURE (clicks_90d) ────────────────────────────────
X_leaky = X_honest.copy()
X_leaky['clicks_90d'] = df_clean['clicks_90d'].values

leaky_maes = []
for train_idx, val_idx in gkf.split(X_leaky, y, groups):
    X_tr, y_tr = X_leaky.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X_leaky.iloc[val_idx], y.iloc[val_idx]
    
    rf_leak = RandomForestRegressor(n_estimators=30, max_depth=8, random_state=42, n_jobs=-1)
    rf_leak.fit(X_tr, y_tr)
    preds_leak = rf_leak.predict(X_va)
    leaky_maes.append(mean_absolute_error(y_va, preds_leak))

mae_leaky_cv = np.mean(leaky_maes)
print(f"2. LEAKED MODEL (GroupKFold MAE):  {mae_leaky_cv:.4f}%  ← Artificial near-zero MAE jump!")
print(f"\n   Score Jump: MAE collapsed from {mae_honest_cv:.4f}% down to {mae_leaky_cv:.4f}%")
print("   DIAGNOSIS: Leaked feature 'clicks_90d' contains the literal numerator of CTR!")

# ── 3. PURGE LEAKY FEATURE ───────────────────────────────────────────────────
del X_leaky
print(f"\n3. PURGED & VERIFIED: 'clicks_90d' removed from feature vector.")
print(f"   Honest Client-Holdout MAE retained: {mae_honest_cv:.4f}%")

1. HONEST MODEL (GroupKFold MAE):  0.7824%


2. LEAKED MODEL (GroupKFold MAE):  0.1332%  ← Artificial near-zero MAE jump!

   Score Jump: MAE collapsed from 0.7824% down to 0.1332%
   DIAGNOSIS: Leaked feature 'clicks_90d' contains the literal numerator of CTR!

3. PURGED & VERIFIED: 'clicks_90d' removed from feature vector.
   Honest Client-Holdout MAE retained: 0.7824%


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Fields Register

| Field | Category | Why Excluded |
|---|---|---|
| `clicks_90d` | **Direct target leakage** | CTR = clicks / impressions × 100. Clicks is the literal numerator of the label — using it makes prediction circular and trivially perfect, as demonstrated by the leakage experiment above. |
| `clicks_last_30d` | **Direct target leakage** | Same numerator relationship as `clicks_90d` but for the most recent 30-day sub-window. |
| `clicks_prev_30d` | **Direct target leakage** | Same numerator relationship for the prior 30-day sub-window. |
| `trend_direction` | **Cross-lane label source** | This is the label source for the Content Refresh lane (`is_declining_label = trend_direction == "down"`). Using it here imports a different lane's target signal. Also derived from change in impressions/clicks which correlates with our CTR target. |
| `trend_pct` | **Cross-lane label source** | The numeric source behind `trend_direction`. Derived from `(impressions_last_30d - impressions_prev_30d) / impressions_prev_30d`. Correlated with the CTR target through shared click/impression denominators. |
| `is_declining_label` | **Cross-lane label** | Binary label derived from `trend_direction`; it is the literal target for a different lane. |
| `content_id` | **Pseudonymized ID** | An opaque hash used only for joining and grouping. Learning from content IDs would memorize individual pages rather than learning generalizable patterns. |
| `client_id` | **Pseudonymized ID** | An opaque hash used for client-holdout splits (`GroupKFold`). Learning from client IDs memorizes client baseline CTR, preventing generalization to unseen clients. |
| `engagement_rate` | **Multi-system denominator mismatch** | GA4 metric that can exceed 100% due to different session-counting methodology. Not safe as a direct feature without careful normalisation (can be revisited in later modeling weeks). |
| `scroll_rate` | **Multi-system denominator mismatch** | GA4 metric that can exceed 100%. Same issue as `engagement_rate`. |
| `ai_traffic_pct` | **Multi-system denominator mismatch** | GA4 metric that can exceed 100%. Same issue. |
| `provider_used` | **Product decision** | Which content provider generated the page — this is a business process flag, not an observable search signal. |
| `model_used` | **Product decision** | Which AI model generated the content — same reasoning as `provider_used`. |

In [4]:
# Verify that none of the excluded fields made it into the final feature vector
excluded_fields_list = [
    'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d',
    'trend_direction', 'trend_pct', 'is_declining_label',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'provider_used', 'model_used'
]

leaked_features_found = [f for f in excluded_fields_list if f in feature_cols]

if len(leaked_features_found) > 0:
    print(f"❌ LEAKAGE DETECTED! The following excluded fields are in your feature vector: {leaked_features_found}")
else:
    print("✓ PASS: None of the excluded fields are present in the final feature vector.")

✓ PASS: None of the excluded fields are present in the final feature vector.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.